In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
!pip install -q datasets transformers scikit-learn

In [8]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")
train = ds["train"].shuffle(seed=42).select(range(5000))
test = ds["test"].shuffle(seed=42).select(range(1000))
print(train[0]["text"][:200], train[0]["label"])  # label 1 = positive, 0 = negative

There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F 1


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(train["text"])
Xte = vec.transform(test["text"])

lr = LogisticRegression(max_iter=1000).fit(Xtr, train["label"])
lr_preds = lr.predict(Xte)
print("Baseline accuracy:", accuracy_score(test["label"], lr_preds))

Baseline accuracy: 0.849


In [7]:
from transformers import pipeline

clf = pipeline("sentiment-analysis",
               model="distilbert-base-uncased-finetuned-sst-2-english",
               truncation=True, max_length=512, device=0)

out = clf(list(test["text"]), batch_size=32)
pre_preds = [1 if o["label"] == "POSITIVE" else 0 for o in out]
print("Pretrained accuracy:", accuracy_score(test["label"], pre_preds))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Pretrained accuracy: 0.881


In [9]:
import numpy as np
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)

def encode(batch):
    return tok(batch["text"], truncation=True, max_length=256)

tr = train.map(encode, batched=True)
te = test.map(encode, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)

args = TrainingArguments(output_dir="out", num_train_epochs=2,
                         per_device_train_batch_size=16,
                         eval_strategy="epoch", report_to="none")

def metrics(p):
    return {"accuracy": float((np.argmax(p.predictions, axis=1) == p.label_ids).mean())}

trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=te,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

... (4 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.410268,0.835000
2,0.306808,0.325329,0.897000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=626, training_loss=0.27536766064433627, metrics={'train_runtime': 246.7578, 'train_samples_per_second': 40.526, 'train_steps_per_second': 2.537, 'total_flos': 662336993280000.0, 'train_loss': 0.27536766064433627, 'epoch': 2.0})